In [117]:
import pandas as pd
import itertools

tea_df = pd.read_csv(r"H:\Projects\student-management-system\data\3.teachers_data.csv")
cls_df=  pd.read_csv(r"H:\Projects\student-management-system\data\4.classes.csv")
sub_df=  pd.read_csv(r"H:\Projects\student-management-system\data\6.subjects.csv")
gra_df=  pd.read_csv(r"H:\Projects\student-management-system\data\1.grades.csv")
print("teachers columns:", tea_df.columns.tolist())
print("classes columns:", cls_df.columns.tolist())
print("subjects columns:", sub_df.columns.tolist())
print("Grade columns:", gra_df.columns.tolist())

sub_df.head()

teachers columns: ['id', 'username', 'name', 'email', 'phone', 'address', 'img', 'bloodType', 'gender', 'dob', 'classId', 'clerk_id']
classes columns: ['id', 'name', 'section', 'gradeId', 'supervisorId']
subjects columns: ['id', 'name', 'gradeIds']
Grade columns: ['id', 'level']


,id,name,gradeIds
0,1,BIOLOGY,"9,10,11,12,13"
1,2,CHEMISTRY,"9,10,11,12,13"
2,3,CLASS ACTIVITIES,1
3,4,COMPUTER,"4,5,6,7,8,9,10,11,12,13"
4,5,DRAWING,"2,3,4,5,6,7,8,9,10,11"


In [118]:
DAYS = ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY"]

PERIODS_BY_DAY = {
    "MONDAY":    ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "TUESDAY":   ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "WEDNESDAY": ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "THURSDAY":  ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "FRIDAY":    ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "SATURDAY":  ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4"],
}

subjects_expanded = (
    sub_df
    .assign(gradeId=sub_df["gradeIds"].astype(str).str.split(","))
    .explode("gradeId")
)

subjects_expanded["gradeId"] = subjects_expanded["gradeId"].astype(int)


In [119]:
import random

lessons = []
lesson_id = 1

for _, cls in cls_df.iterrows():
    class_id = cls["id"]
    grade_id = cls["gradeId"]

    # Teacher assigned to class
    teacher_row = tea_df[tea_df["classId"] == class_id]
    if teacher_row.empty:
        continue

    teacher_id = teacher_row.iloc[0]["id"]

    # Subjects for grade (cycled)
    subjects = subjects_expanded[
        subjects_expanded["gradeId"] == grade_id
    ]["name"].tolist()

    subject_index = 0

    for day in DAYS:
        periods = PERIODS_BY_DAY[day]

        # Random FREE period per day
        free_period = random.choice(periods)

        for period in periods:
            if period == free_period:
                lessons.append({
                    "id": lesson_id,
                    "gradeId": grade_id,
                    "classId": class_id,
                    "subject": "",
                    "teacherId": None,
                    "day": day,
                    "period": period
                })
            else:
                lessons.append({
                    "id": lesson_id,
                    "gradeId": grade_id,
                    "classId": class_id,
                    "subject": subjects[subject_index % len(subjects)],
                    "teacherId": teacher_id,
                    "day": day,
                    "period": period
                })
                subject_index += 1

            lesson_id += 1


In [120]:
lessons_df = pd.DataFrame(lessons)
lessons_df.head(10)


,id,gradeId,classId,subject,teacherId,day,period
0,1,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD1
1,2,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD2
2,3,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD3
3,4,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD4
4,5,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD5
5,6,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD6
6,7,1,1,,None,MONDAY,PERIOD7
7,8,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD8
8,9,1,1,CLASS ACTIVITIES,staff_ks_063,TUESDAY,PERIOD1
9,10,1,1,CLASS ACTIVITIES,staff_ks_063,TUESDAY,PERIOD2


In [121]:
sub_df.head(1)

,id,name,gradeIds
0,1,BIOLOGY,"9,10,11,12,13"


In [122]:
sub_df.rename(columns={"name": "subject"}, inplace=True)

In [123]:
sub_df.head(5)

,id,subject,gradeIds
0,1,BIOLOGY,"9,10,11,12,13"
1,2,CHEMISTRY,"9,10,11,12,13"
2,3,CLASS ACTIVITIES,1
3,4,COMPUTER,"4,5,6,7,8,9,10,11,12,13"
4,5,DRAWING,"2,3,4,5,6,7,8,9,10,11"


In [124]:
lessons_df = pd.merge(lessons_df,sub_df, on="subject", how="left")
lessons_df

,id_x,gradeId,classId,subject,teacherId,day,period,id_y,gradeIds
0,1,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD1,3.0,1
1,2,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD2,3.0,1
2,3,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD3,3.0,1
3,4,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD4,3.0,1
4,5,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD5,3.0,1
...,...,...,...,...,...,...,...,...,...
1887,1888,13,44,ENGLISH II,staff_ks_062,FRIDAY,PERIOD8,7.0,"4,5,6,7,8,9,10,11,12,13"
1888,1889,13,44,GEOGRAPHY,staff_ks_062,SATURDAY,PERIOD1,13.0,"9,10,11,12,13"
1889,1890,13,44,HISTORY & CIVICS,staff_ks_062,SATURDAY,PERIOD2,15.0,"9,10,11,12,13"
1890,1891,13,44,II LANGUAGE (TELUGU/HINDI),staff_ks_062,SATURDAY,PERIOD3,16.0,"3,4,5,6,7,8,9,10,11,12,13"


In [125]:
lessons_df.rename(columns={"id_x" : "id", "id_y" : "subjectId"}, inplace=True)

In [126]:
lessons_df

,id,gradeId,classId,subject,teacherId,day,period,subjectId,gradeIds
0,1,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD1,3.0,1
1,2,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD2,3.0,1
2,3,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD3,3.0,1
3,4,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD4,3.0,1
4,5,1,1,CLASS ACTIVITIES,staff_ks_063,MONDAY,PERIOD5,3.0,1
...,...,...,...,...,...,...,...,...,...
1887,1888,13,44,ENGLISH II,staff_ks_062,FRIDAY,PERIOD8,7.0,"4,5,6,7,8,9,10,11,12,13"
1888,1889,13,44,GEOGRAPHY,staff_ks_062,SATURDAY,PERIOD1,13.0,"9,10,11,12,13"
1889,1890,13,44,HISTORY & CIVICS,staff_ks_062,SATURDAY,PERIOD2,15.0,"9,10,11,12,13"
1890,1891,13,44,II LANGUAGE (TELUGU/HINDI),staff_ks_062,SATURDAY,PERIOD3,16.0,"3,4,5,6,7,8,9,10,11,12,13"


In [127]:
lessons_df = lessons_df.drop(columns=["subject", "gradeIds"])
lessons_df

,id,gradeId,classId,teacherId,day,period,subjectId
0,1,1,1,staff_ks_063,MONDAY,PERIOD1,3.0
1,2,1,1,staff_ks_063,MONDAY,PERIOD2,3.0
2,3,1,1,staff_ks_063,MONDAY,PERIOD3,3.0
3,4,1,1,staff_ks_063,MONDAY,PERIOD4,3.0
4,5,1,1,staff_ks_063,MONDAY,PERIOD5,3.0
...,...,...,...,...,...,...,...
1887,1888,13,44,staff_ks_062,FRIDAY,PERIOD8,7.0
1888,1889,13,44,staff_ks_062,SATURDAY,PERIOD1,13.0
1889,1890,13,44,staff_ks_062,SATURDAY,PERIOD2,15.0
1890,1891,13,44,staff_ks_062,SATURDAY,PERIOD3,16.0


In [128]:
lessons_df.isnull().sum()

id             0
gradeId        0
classId        0
teacherId    258
day            0
period         0
subjectId    258
dtype: int64

In [129]:
lessons_df = lessons_df.dropna(subset=["teacherId"])

In [130]:
lessons_df = lessons_df.reset_index(drop=True)

In [131]:
lessons_df["id"] = range(1, len(lessons_df) + 1)
lessons_df

,id,gradeId,classId,teacherId,day,period,subjectId
0,1,1,1,staff_ks_063,MONDAY,PERIOD1,3.0
1,2,1,1,staff_ks_063,MONDAY,PERIOD2,3.0
2,3,1,1,staff_ks_063,MONDAY,PERIOD3,3.0
3,4,1,1,staff_ks_063,MONDAY,PERIOD4,3.0
4,5,1,1,staff_ks_063,MONDAY,PERIOD5,3.0
...,...,...,...,...,...,...,...
1629,1630,13,44,staff_ks_062,FRIDAY,PERIOD6,6.0
1630,1631,13,44,staff_ks_062,FRIDAY,PERIOD8,7.0
1631,1632,13,44,staff_ks_062,SATURDAY,PERIOD1,13.0
1632,1633,13,44,staff_ks_062,SATURDAY,PERIOD2,15.0


In [132]:
lessons_df['subjectId'] = lessons_df['subjectId'].astype(int)

In [133]:
lessons_df

,id,gradeId,classId,teacherId,day,period,subjectId
0,1,1,1,staff_ks_063,MONDAY,PERIOD1,3
1,2,1,1,staff_ks_063,MONDAY,PERIOD2,3
2,3,1,1,staff_ks_063,MONDAY,PERIOD3,3
3,4,1,1,staff_ks_063,MONDAY,PERIOD4,3
4,5,1,1,staff_ks_063,MONDAY,PERIOD5,3
...,...,...,...,...,...,...,...
1629,1630,13,44,staff_ks_062,FRIDAY,PERIOD6,6
1630,1631,13,44,staff_ks_062,FRIDAY,PERIOD8,7
1631,1632,13,44,staff_ks_062,SATURDAY,PERIOD1,13
1632,1633,13,44,staff_ks_062,SATURDAY,PERIOD2,15


In [134]:
lessons_df.to_csv(
    r"H:\Projects\student-management-system\data\8.lessons.csv",
    index=False
)
